In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
!pip install -U "bitsandbytes>=0.46.1"

In [ ]:
import bitsandbytes as bnb

print("bitsandbytes version:", bnb.__version__)

bitsandbytes version: 0.50.2


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-3B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model directly in FP16
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))

print(
    "GPU memory used:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
GPU: Tesla T4
GPU memory used: 5.75 GB


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the internet for current information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

messages = [
    {
        "role": "user",
        "content": "Find the latest LangGraph release. Use the search_web tool."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "search_web", "description": "Search the internet for current information.", "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "The search query"}}, "required": ["query"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Find the latest LangGraph release. Use the search_web tool.<|im_end|>
<|im_start|>assistant



In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

generated = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=False
)

print(generated)

<tool_call>
{"name": "search_web", "arguments": {"query": "latest LangGraph release"}}
</tool_call><|im_end|>


In [ ]:
import json

tool_response = json.loads(
    generated.split("<tool_call>")[1].split("</tool_call>")[0]
)

tool_name = tool_response["name"]
tool_args = tool_response["arguments"]

print("Tool name:", tool_name)
print("Tool arguments:", tool_args)

Tool name: search_web
Tool arguments: {'query': 'latest LangGraph release'}


In [ ]:
def search_web(query: str):
    return """
    Latest LangGraph release information:

    Version: 1.2.20
    Framework: LangGraph
    Description: A framework for building stateful,
    multi-agent applications using graphs.
    """

# Execute the function again
result = search_web(**tool_args)

print("ACTUAL TOOL RESULT:")
print(result)

ACTUAL TOOL RESULT:

    Latest LangGraph release information:

    Version: 1.2.20
    Framework: LangGraph
    Description: A framework for building stateful,
    multi-agent applications using graphs.
    


In [ ]:
messages = [
    {
        "role": "user",
        "content": "Find the latest LangGraph release. Use the search_web tool."
    },
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "type": "function",
                "function": {
                    "name": tool_name,
                    "arguments": json.dumps(tool_args)
                }
            }
        ]
    },
    {
        "role": "tool",
        "name": tool_name,
        "content": result
    }
]

print(messages)

[{'role': 'user', 'content': 'Find the latest LangGraph release. Use the search_web tool.'}, {'role': 'assistant', 'content': None, 'tool_calls': [{'type': 'function', 'function': {'name': 'search_web', 'arguments': '{"query": "latest LangGraph release"}'}}]}, {'role': 'tool', 'name': 'search_web', 'content': '\n    Latest LangGraph release information:\n\n    Version: 1.2.20\n    Framework: LangGraph\n    Description: A framework for building stateful,\n    multi-agent applications using graphs.\n    '}]


In [ ]:
final_prompt = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    final_prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

final_answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(final_answer)

The latest version of LangGraph is 1.2.20. It's a framework designed for building stateful, multi-agent applications using graphs.
